# Thai Air Intelligence — Six-model PM2.5 Next-day Trainer
Trains and compares Random Forest, AdaBoost, Gradient Boosting, XGBoost, LightGBM, and CatBoost per province with chronological rolling validation and an untouched final holdout.

> Required Colab Secrets: `SUPABASE_URL` and `SUPABASE_SERVICE_ROLE_KEY` only. `ACTIVATE=True`: eligible provinces promote `ensemble6-pm25-v3`; every other Isan province temporarily reverts to `persist-revert-v2`.


In [ ]:
# 1. Install only packages missing from Colab; do not replace NumPy/Pandas/scikit-learn
%pip install -q --upgrade --upgrade-strategy only-if-needed supabase==2.31.0 xgboost==2.1.4 lightgbm==4.6.0 catboost==1.2.10 joblib==1.4.2


In [ ]:
# 2. Load and validate the two required Colab Secrets without printing their values

from google.colab import userdata
import json
import math
import uuid
import zipfile
import platform
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from supabase import create_client

REQUIRED_SECRETS = [
    "SUPABASE_URL",
    "SUPABASE_SERVICE_ROLE_KEY",
]

secrets = {
    name: (userdata.get(name) or "").strip()
    for name in REQUIRED_SECRETS
}

missing = [
    name
    for name, value in secrets.items()
    if not value
]

if missing:
    raise RuntimeError(
        f"Missing Colab Secrets: {missing}"
    )

SUPABASE_URL = secrets["SUPABASE_URL"].rstrip("/")
SUPABASE_SERVICE_ROLE_KEY = secrets["SUPABASE_SERVICE_ROLE_KEY"]

# ตรวจสอบ URL
if not (
    SUPABASE_URL.startswith("https://")
    and ".supabase.co" in SUPABASE_URL
):
    raise ValueError(
        "SUPABASE_URL must look like "
        "https://<project-ref>.supabase.co"
    )

# ตรวจสอบ placeholder
key_lower = SUPABASE_SERVICE_ROLE_KEY.lower()

if key_lower.startswith(
    ("your_", "replace-", "paste-", "example")
):
    raise ValueError(
        "SUPABASE_SERVICE_ROLE_KEY is still a placeholder"
    )

# ป้องกันการใส่ public key ผิดช่อง
if key_lower.startswith(
    ("sb_publishable_", "sb_anon_")
):
    raise ValueError(
        "ใส่คีย์ผิดประเภท: ต้องใช้ sb_secret_ "
        "หรือ legacy service_role key เท่านั้น"
    )

# รองรับทั้ง Secret Key รุ่นใหม่และ legacy service-role JWT
if key_lower.startswith("sb_secret_"):
    key_type = "modern sb_secret key"
elif SUPABASE_SERVICE_ROLE_KEY.startswith("eyJ"):
    key_type = "legacy service_role JWT"
else:
    key_type = "server key (format will be verified by Supabase)"

print("✅ Colab Secrets loaded successfully")
print("Supabase key type:", key_type)
print(
    "Runtime:",
    {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
    },
)

In [ ]:
# 3. Connect to Supabase and verify the training view
sb = create_client(SUPABASE_URL, SUPABASE_SERVICE_ROLE_KEY)

probe = (
    sb.table("training_daily_summary_v2")
    .select("province_id,date,pm25_mean,trusted_hours,trusted_sources")
    .limit(1)
    .execute()
)

if not probe.data:
    raise RuntimeError("training_daily_summary_v2 has no usable rows")

print("✅ Training view reachable")
print("Sample:", probe.data[0])

In [ ]:
# 4. Configuration
REGISTER = True
ACTIVATE = True
MIN_ROWS = 180
CV_SPLITS = 5
ALLOWED_SOURCES = {"open-meteo"}
MODEL_NAME = "ensemble6-pm25-v3"  # model_registry varchar(20)
PERSIST_MODEL = "persist-revert-v2"
MODEL_DISPLAY_NAME = "PM2.5 six-model comparison v3"
SKILL_THRESHOLD = 0.05
FINAL_MIN_ROWS = 30
HOLDOUT_FRAC = 0.20
RANDOM_STATE = 42
N_JOBS = -1

FEATURES = [
    "pm25_mean", "pm25_lag_1d", "pm25_lag_3d", "pm25_lag_7d", "pm25_roll7",
    "neighbor_pm25_avg", "regional_pm25_avg", "temp_mean",
    "humidity_mean", "wind_speed_mean", "precip_total",
    "hotspot_count", "total_frp", "month", "day_of_week",
    "is_burning_season", "is_dry_season",
]

assert len(MODEL_NAME) <= 20
RUN_ID = str(uuid.uuid4())
GIT_SHA = "colab-manual"
print({
    "run_id": RUN_ID,
    "model_name": MODEL_NAME,
    "register": REGISTER,
    "activate": ACTIVATE,
    "min_rows": MIN_ROWS,
    "cv_splits": CV_SPLITS,
})


In [ ]:
# 5. Fetch data with pagination
def fetch_all(table, page_size=1000):
    rows=[]; start=0
    while True:
        end=start+page_size-1
        resp=sb.table(table).select("*").range(start,end).order("province_id").order("date").execute()
        batch=resp.data or []; rows.extend(batch)
        if len(batch)<page_size: break
        start += page_size
    return pd.DataFrame(rows)
raw=fetch_all("training_daily_summary_v2")
print("Fetched rows:", len(raw))

In [ ]:
# 6. Enforce trusted observations and show quality per province
df = raw.copy()
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["trusted_hours"] = pd.to_numeric(df["trusted_hours"], errors="coerce").fillna(0)
def trusted_source_ok(value):
    values = value if isinstance(value, list) else []
    return bool(ALLOWED_SOURCES.intersection(values))
df = df[df["trusted_hours"].ge(18) & df["trusted_sources"].map(trusted_source_ok)]
df = df.dropna(subset=["province_id", "date", "pm25_mean"]).sort_values(["province_id", "date"])
quality = (df.groupby("province_id").agg(rows=("date", "size"), first_date=("date", "min"), last_date=("date", "max"), pm25_mean=("pm25_mean", "mean"), min_trusted_hours=("trusted_hours", "min")).reset_index())
display(quality)

In [ ]:
# 7. Prepare features and strictly consecutive next-day target
def prepare_one(g):
    g=g.sort_values("date").copy(); g["next_date"]=g["date"].shift(-1); g["target_pm25_next_day"]=g["pm25_mean"].shift(-1)
    g["is_consecutive_target"]=(pd.to_datetime(g["next_date"])-pd.to_datetime(g["date"])).dt.days.eq(1)
    need=FEATURES+["target_pm25_next_day"]
    return g[g["is_consecutive_target"]].dropna(subset=need)
sets={pid: prepare_one(g) for pid,g in df.groupby("province_id")}
print({pid: len(v) for pid,v in sets.items()})

In [ ]:
# 8. Rolling-origin CV for all six models — chronological, no shuffle
from sklearn.ensemble import (
    RandomForestRegressor,
    AdaBoostRegressor,
    GradientBoostingRegressor,
)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

MODEL_ORDER = [
    "random_forest",
    "adaboost",
    "gradient_boosting",
    "xgboost",
    "lightgbm",
    "catboost",
]


def metrics(y_true, y_pred, persistence_pred):
    mae = mean_absolute_error(y_true, y_pred)
    persistence_mae = mean_absolute_error(y_true, persistence_pred)
    return {
        "mae": float(mae),
        "rmse": float(math.sqrt(mean_squared_error(y_true, y_pred))),
        "r2": float(r2_score(y_true, y_pred)) if len(y_true) > 1 else 0.0,
        "persistence_mae": float(persistence_mae),
        "skill": (
            float((persistence_mae - mae) / persistence_mae)
            if persistence_mae > 0 else -1.0
        ),
    }


def rolling_splits(n_rows, splits=CV_SPLITS):
    final_size = max(FINAL_MIN_ROWS, math.ceil(n_rows * HOLDOUT_FRAC))
    development_size = n_rows - final_size
    minimum_train_size = max(60, MIN_ROWS // 2)
    available_validation = development_size - minimum_train_size
    if available_validation < splits:
        return
    validation_size = max(1, available_validation // splits)
    for fold in range(splits):
        train_end = minimum_train_size + fold * validation_size
        validation_end = (
            development_size
            if fold == splits - 1
            else min(development_size, train_end + validation_size)
        )
        if validation_end <= train_end:
            continue
        yield np.arange(0, train_end), np.arange(train_end, validation_end)


def make_models():
    return {
        "random_forest": RandomForestRegressor(
            n_estimators=300,
            max_depth=8,
            min_samples_leaf=3,
            max_features=0.8,
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS,
        ),
        "adaboost": AdaBoostRegressor(
            n_estimators=300,
            learning_rate=0.03,
            loss="square",
            random_state=RANDOM_STATE,
        ),
        "gradient_boosting": GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.03,
            max_depth=3,
            min_samples_leaf=3,
            subsample=0.9,
            loss="huber",
            random_state=RANDOM_STATE,
        ),
        "xgboost": XGBRegressor(
            n_estimators=400,
            max_depth=4,
            learning_rate=0.03,
            min_child_weight=3,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.0,
            random_state=RANDOM_STATE,
            objective="reg:squarederror",
            n_jobs=N_JOBS,
        ),
        "lightgbm": LGBMRegressor(
            n_estimators=400,
            max_depth=5,
            num_leaves=24,
            learning_rate=0.03,
            min_child_samples=15,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.0,
            random_state=RANDOM_STATE,
            verbose=-1,
            n_jobs=N_JOBS,
        ),
        "catboost": CatBoostRegressor(
            iterations=400,
            depth=5,
            learning_rate=0.03,
            loss_function="MAE",
            random_seed=RANDOM_STATE,
            verbose=False,
            thread_count=N_JOBS,
            allow_writing_files=False,
        ),
    }


CV_COLUMNS = [
    "province_id", "fold", "model", "mae", "rmse", "r2",
    "persistence_mae", "skill",
]
cv_rows = []
skipped_rows = {}

for province_id, province_data in sets.items():
    row_count = len(province_data)
    if row_count < MIN_ROWS:
        skipped_rows[province_id] = row_count
        continue

    X = province_data[FEATURES].to_numpy(dtype=float)
    y = province_data["target_pm25_next_day"].to_numpy(dtype=float)
    persistence = province_data["pm25_mean"].to_numpy(dtype=float)
    province_splits = list(rolling_splits(row_count))

    if not province_splits:
        skipped_rows[province_id] = row_count
        print(f"⚠️ {province_id}: {row_count} rows are insufficient for rolling CV")
        continue

    print(f"Training CV: {province_id} ({row_count} rows)")
    for fold, (train_indices, validation_indices) in enumerate(province_splits, 1):
        for model_name, model in make_models().items():
            model.fit(X[train_indices], y[train_indices])
            prediction = model.predict(X[validation_indices])
            cv_rows.append({
                "province_id": province_id,
                "fold": fold,
                "model": model_name,
                **metrics(
                    y[validation_indices],
                    prediction,
                    persistence[validation_indices],
                ),
            })

cv = pd.DataFrame(cv_rows, columns=CV_COLUMNS)
if cv.empty:
    print(f"⚠️ No province has at least {MIN_ROWS} usable rows")
    cv_summary = pd.DataFrame(columns=CV_COLUMNS)
    model_summary = pd.DataFrame()
    display(pd.DataFrame([
        {
            "province_id": pid,
            "usable_rows": count,
            "required_rows": MIN_ROWS,
            "missing_rows": max(0, MIN_ROWS - count),
        }
        for pid, count in sorted(skipped_rows.items())
    ]))
else:
    cv_summary = (
        cv.groupby(["province_id", "model"], as_index=False)
        .agg(
            folds=("fold", "nunique"),
            mae=("mae", "mean"),
            rmse=("rmse", "mean"),
            r2=("r2", "mean"),
            persistence_mae=("persistence_mae", "mean"),
            skill=("skill", "mean"),
        )
    )
    model_summary = (
        cv_summary.groupby("model", as_index=False)
        .agg(
            provinces=("province_id", "nunique"),
            mean_mae=("mae", "mean"),
            median_mae=("mae", "median"),
            mean_rmse=("rmse", "mean"),
            mean_r2=("r2", "mean"),
            mean_skill=("skill", "mean"),
            positive_skill_rate=("skill", lambda s: float((s > 0).mean())),
        )
        .sort_values(["mean_skill", "mean_mae"], ascending=[False, True])
        .reset_index(drop=True)
    )
    model_summary.insert(0, "cv_rank", np.arange(1, len(model_summary) + 1))
    display(cv_summary.sort_values(["province_id", "skill"], ascending=[True, False]))
    print("Global rolling-CV ranking")
    display(model_summary)


In [ ]:
# 9. Final holdout comparison, select winner by CV, then distil runtime surrogate
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
import joblib

ART = Path("artifacts") / RUN_ID
ART.mkdir(parents=True, exist_ok=True)
results = []
final_model_rows = []

for pid, d in sets.items():
    if len(d) < MIN_ROWS:
        continue
    final_n = max(FINAL_MIN_ROWS, math.ceil(len(d) * HOLDOUT_FRAC))
    train = d.iloc[:-final_n]
    test = d.iloc[-final_n:]

    province_cv = cv_summary[cv_summary.province_id.eq(pid)].sort_values(
        ["skill", "mae"], ascending=[False, True]
    )
    if province_cv.empty:
        continue
    teacher_name = province_cv.iloc[0]["model"]

    Xtr = train[FEATURES].to_numpy(float)
    ytr = train.target_pm25_next_day.to_numpy(float)
    Xte = test[FEATURES].to_numpy(float)
    yte = test.target_pm25_next_day.to_numpy(float)
    bte = test.pm25_mean.to_numpy(float)

    fitted_models = {}
    native_metrics = {}
    for model_name, model in make_models().items():
        model.fit(Xtr, ytr)
        prediction = model.predict(Xte)
        score = metrics(yte, prediction, bte)
        fitted_models[model_name] = model
        native_metrics[model_name] = score
        final_model_rows.append({
            "province_id": pid,
            "model": model_name,
            "selected_by_cv": model_name == teacher_name,
            "final_rows": final_n,
            **score,
        })

    teacher = fitted_models[teacher_name]
    selected_native_metrics = native_metrics[teacher_name]

    # Runtime currently consumes a standardized linear surrogate from model_registry.
    scaler = StandardScaler().fit(Xtr)
    ridge = Ridge(alpha=1.0).fit(scaler.transform(Xtr), teacher.predict(Xtr))
    surrogate_prediction = ridge.predict(scaler.transform(Xte))
    surrogate_metrics = metrics(yte, surrogate_prediction, bte)
    residual_p90 = float(np.percentile(np.abs(yte - surrogate_prediction), 90))

    raw_importance = getattr(teacher, "feature_importances_", np.zeros(len(FEATURES)))
    feature_importance = {
        feature: float(value)
        for feature, value in zip(FEATURES, raw_importance)
    }
    native_path = str(ART / f"{pid}_{teacher_name}.joblib")
    joblib.dump(teacher, native_path)
    manifest = {
        "run_id": RUN_ID,
        "model_name": MODEL_NAME,
        "model_display_name": MODEL_DISPLAY_NAME,
        "province_id": pid,
        "teacher_model": teacher_name,
        "candidate_models": MODEL_ORDER,
        "selection_rule": "highest rolling-CV skill, tie-break lowest rolling-CV MAE",
        "feature_order": FEATURES,
        "native_artifact": native_path,
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
    (ART / f"{pid}_manifest.json").write_text(json.dumps(manifest, indent=2))
    params = {
        **manifest,
        "source": "training_daily_summary_v2",
        "dependency_versions": {"python": platform.python_version()},
        "hyperparameters": teacher.get_params(),
        "metrics": {
            "all_native_final": native_metrics,
            "selected_native_final": selected_native_metrics,
            "surrogate_final": surrogate_metrics,
        },
        "git_sha": GIT_SHA,
        "surrogate": {
            "feature_cols": FEATURES,
            "coefficients": [float(x) for x in ridge.coef_],
            "intercept": float(ridge.intercept_),
            "scaler_mean": [float(x) for x in scaler.mean_],
            "scaler_scale": [float(x) for x in scaler.scale_],
        },
        "upper_residual_q90": residual_p90,
        "feature_importance": feature_importance,
    }
    eligible_flag = (
        final_n >= FINAL_MIN_ROWS
        and surrogate_metrics["skill"] >= SKILL_THRESHOLD
        and surrogate_metrics["mae"] < surrogate_metrics["persistence_mae"]
    )
    results.append({
        "province_id": pid,
        "rows": len(d),
        "final_rows": final_n,
        "teacher": teacher_name,
        "native_mae": selected_native_metrics["mae"],
        "native_skill": selected_native_metrics["skill"],
        "eligible": eligible_flag,
        "params": params,
        "train_start": str(train.date.min()),
        "train_end": str(train.date.max()),
        "test_start": str(test.date.min()),
        "test_end": str(test.date.max()),
        "data_cutoff": str(d.date.max()),
        **surrogate_metrics,
    })

final_comparison = pd.DataFrame(final_model_rows)
res = pd.DataFrame([{k: v for k, v in row.items() if k != "params"} for row in results])

if final_comparison.empty:
    global_final_summary = pd.DataFrame()
else:
    global_final_summary = (
        final_comparison.groupby("model", as_index=False)
        .agg(
            provinces=("province_id", "nunique"),
            mean_mae=("mae", "mean"),
            median_mae=("mae", "median"),
            mean_rmse=("rmse", "mean"),
            mean_r2=("r2", "mean"),
            mean_skill=("skill", "mean"),
            passed_skill_threshold=("skill", lambda s: int((s >= SKILL_THRESHOLD).sum())),
        )
        .sort_values(["mean_skill", "mean_mae"], ascending=[False, True])
        .reset_index(drop=True)
    )
    global_final_summary.insert(0, "final_rank", np.arange(1, len(global_final_summary) + 1))

display(res)


In [ ]:
# 10. Display all-model final comparison, overall ranking, and selected runtime candidates
import matplotlib.pyplot as plt

RESULT_COLUMNS = [
    "province_id", "teacher", "final_rows", "native_mae", "native_skill",
    "mae", "rmse", "r2", "persistence_mae", "skill", "eligible",
]

if res.empty:
    print(f"⚠️ No province has at least {MIN_ROWS} usable rows")
    final_metrics = pd.DataFrame(columns=RESULT_COLUMNS)
    display(final_metrics)
else:
    final_metrics = res[RESULT_COLUMNS].sort_values("province_id").reset_index(drop=True)
    print("Native model comparison on untouched final holdout")
    display(final_comparison.sort_values(
        ["province_id", "skill"], ascending=[True, False]
    ).reset_index(drop=True))
    print("Global final-holdout ranking (diagnostic only; model selection used CV)")
    display(global_final_summary)
    print("CV-selected teacher and runtime-surrogate metrics")
    display(final_metrics)

    plot_data = global_final_summary.sort_values("mean_mae", ascending=True)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].barh(plot_data["model"], plot_data["mean_mae"], color="#2563eb")
    axes[0].set_title("Mean MAE on final holdout (lower is better)")
    axes[0].set_xlabel("PM2.5 MAE")
    skill_data = global_final_summary.sort_values("mean_skill", ascending=True)
    axes[1].barh(skill_data["model"], skill_data["mean_skill"], color="#16a34a")
    axes[1].axvline(SKILL_THRESHOLD, color="#dc2626", linestyle="--", label="threshold")
    axes[1].set_title("Mean skill vs persistence (higher is better)")
    axes[1].set_xlabel("Skill")
    axes[1].legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# 11. Eligible provinces
eligible=[r for r in results if r["eligible"]]
print("Eligible:",[r["province_id"] for r in eligible])

In [ ]:
# 12. Validate registry-compatible model identity
assert len(MODEL_NAME) <= 20
for result in results:
    result["params"]["model_name"] = MODEL_NAME
    result["params"]["run_id"] = str(RUN_ID)
print("MODEL_NAME:", MODEL_NAME, "length:", len(MODEL_NAME))


In [ ]:
# 12. Register candidates directly and safely
def json_safe(value):
    if isinstance(value, np.generic):
        value = value.item()

    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}

    if isinstance(value, (list, tuple, np.ndarray)):
        return [json_safe(v) for v in value]

    if isinstance(value, float) and not math.isfinite(value):
        return None

    return value


if REGISTER and results:
    now = datetime.now(timezone.utc).isoformat()
    saved = 0

    for result in results:
        row = json_safe({
            "run_id": str(RUN_ID),
            "model_name": str(MODEL_NAME),
            "province_id": str(result["province_id"]),
            "trained_at": now,
            "training_rows": int(result["rows"]),
            "mae": round(float(result["mae"]), 4),
            "rmse": round(float(result["rmse"]), 4),
            "r2": round(float(result["r2"]), 4),
            "is_active": False,
            "model_params": result["params"],
            "data_cutoff": result["data_cutoff"],
            "train_start": result["train_start"],
            "train_end": result["train_end"],
            "test_start": result["test_start"],
            "test_end": result["test_end"],
            "source": "training_daily_summary_v2",
        })

        # ตรวจ payload ก่อนส่ง
        assert row["model_name"], "model_name is empty"
        assert row["province_id"], "province_id is empty"
        assert row["run_id"], "run_id is empty"
        json.dumps(row, allow_nan=False)

        response = (
            sb.table("model_registry")
            .upsert(
                row,
                on_conflict="model_name,province_id,run_id",
            )
            .execute()
        )

        saved += 1
        print(
            f"✅ [{saved}/{len(results)}] "
            f"{row['province_id']} registered"
        )

    print("✅ Registered inactive candidates:", saved)

else:
    print(
        "Nothing registered: REGISTER is disabled "
        "or no province passed MIN_ROWS."
    )

In [ ]:
# 13. Activate eligible ensemble candidates; revert all others to persistence
if ACTIVATE:
    province_rows = sb.table("isan_provinces").select("province_id").execute().data or []
    target_provinces = sorted({str(row["province_id"]) for row in province_rows})
    eligible_ids = {str(row["province_id"]) for row in results if row["eligible"]}
    ensemble_ids = sorted(set(target_provinces).intersection(eligible_ids))
    fallback_ids = sorted(set(target_provinces).difference(eligible_ids))

    # Preflight every target before changing any active model.
    ensemble_rows = (
        sb.table("model_registry")
        .select("province_id")
        .eq("model_name", MODEL_NAME)
        .eq("run_id", RUN_ID)
        .execute()
        .data or []
    )
    persist_rows = (
        sb.table("model_registry")
        .select("province_id")
        .eq("model_name", PERSIST_MODEL)
        .execute()
        .data or []
    )
    registered_ensemble = {str(row["province_id"]) for row in ensemble_rows}
    registered_persist = {str(row["province_id"]) for row in persist_rows}
    missing_ensemble = sorted(set(ensemble_ids).difference(registered_ensemble))
    missing_persist = sorted(set(fallback_ids).difference(registered_persist))
    if missing_ensemble or missing_persist:
        raise RuntimeError({
            "missing_ensemble_candidates": missing_ensemble,
            "missing_persist_candidates": missing_persist,
        })

    activation_plan = pd.DataFrame([
        {
            "province_id": province_id,
            "eligible": province_id in eligible_ids,
            "activate_model": MODEL_NAME if province_id in eligible_ids else PERSIST_MODEL,
        }
        for province_id in target_provinces
    ])
    display(activation_plan)

    for row in activation_plan.to_dict("records"):
        is_eligible = bool(row["eligible"])
        activated = sb.rpc("fn_activate_model", {
            "p_province_id": row["province_id"],
            "p_run_id": RUN_ID if is_eligible else None,
            "p_model_name": MODEL_NAME if is_eligible else PERSIST_MODEL,
        }).execute().data
        print(row["province_id"], activated)
else:
    print("ACTIVATE is False; no production active model changed.")


In [ ]:
# 14. Verify one active model per province
active=sb.table("model_registry").select("province_id,model_name,run_id,is_active").eq("is_active",True).execute().data or []
chk=pd.DataFrame(active).groupby("province_id").size().reset_index(name="active_count") if active else pd.DataFrame(columns=["province_id","active_count"])
display(chk); assert chk.empty or chk.active_count.max()<=1

In [ ]:
# 15. Prepare artifacts ZIP for download
zip_path=Path(f"pm25_artifacts_{RUN_ID}.zip")
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for path in ART.glob("*"): z.write(path, path.relative_to(ART.parent))
print("ZIP ready:", zip_path)
from google.colab import files
files.download(str(zip_path))